# kNN-based ID / OOD builder (mild + hard)

This notebook loads the *full* cleaned Craigslist dataset, reuses the training preprocessing (numeric z-scales, one-hot levels, hashing) from the selected run, and builds two joint-density OOD splits via kNN distance to the training set:

- **mild OOD**: 80–95th percentile of kNN distance on the test split
- **hard OOD**: top 5% of kNN distance on the test split

It writes three DataFrames (and CSVs) to `datasets/knn_ood/`:
- `craigslist_ood_knn_mild.csv`
- `craigslist_ood_knn_hard.csv`
- (optional) `craigslist_id_knn_in.csv` for in-density ID control (bottom 80%).

It also prints basic metrics: counts, kNN score ranges, and summary stats for the OOD selections.

In [1]:
import csv
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import yaml

# -----------------
# USER KNOBS
# -----------------
FULL_CSV = Path('/home/canelo/uncertainty_quantification/datasets/craigslist_cleaned_FULL.csv')
BASE_RUN_KEY = 'point'          # which run to take preproc_meta from (fallbacks to lpl/gau)
PROJ_DIM = 32                   # random projection dim for kNN distance
KNN_K = 10
REF_TRAIN_N = 50_000            # sample of train rows for reference
CHUNK_ROWS = 50_000
MILD_Q = (0.80, 0.95)
HARD_Q = (0.95, 1.00)
WRITE_ID_IN = True
ID_IN_Q = (0.00, 0.80)
OUT_DIR = Path('/home/canelo/uncertainty_quantification/datasets/knn_ood')
RANDOM_SEED = 0
# -----------------

rng = np.random.default_rng(RANDOM_SEED)

def _find_training_dir_for_run_tag(run_tag: str) -> Path:
    root = Path('/home/canelo/uncertainty_quantification/outputs/trainings').resolve()
    hits = [p for p in root.rglob(run_tag) if p.is_dir() and p.name == run_tag]
    if not hits:
        raise FileNotFoundError(f'No training dir named {run_tag!r} under {root}')
    hits.sort(key=lambda p: p.stat().st_mtime)
    return hits[-1]

def _hash_bucket_md5(s: str, n: int) -> int:
    h = hashlib.md5(s.encode('utf-8')).hexdigest()
    return int(h, 16) % n

def _add_derived(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    df = df.copy()
    if target_col in df.columns:
        df = df[df[target_col].notna()].copy()
    if 'lat' in df.columns:
        df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
    if 'long' in df.columns:
        df['long'] = pd.to_numeric(df['long'], errors='coerce')
    if 'year' in df.columns and 'odometer' in df.columns:
        cur_year = pd.Timestamp.now().year
        year_num = pd.to_numeric(df['year'], errors='coerce')
        odo_num = pd.to_numeric(df['odometer'], errors='coerce').fillna(0.0)
        age = (cur_year - year_num).clip(lower=0)
        df['age_years'] = age
        df['log_odometer'] = np.log1p(odo_num)
        df['mileage_per_year'] = odo_num / np.where(age.to_numpy(float) > 0, age.to_numpy(float), 1.0)
    return df

# --- resolve run + metadata ---
from pathlib import Path as _P
RUN_DIRS = {}
EVAL_ROOT = Path('/home/canelo/uncertainty_quantification/outputs/evals')
RUN_TAG = '_m_'
def _pick_latest_dir(glob_pat: str):
    matches = sorted(EVAL_ROOT.glob(glob_pat))
    return matches[-1] if matches else None
RUN_DIRS['point'] = _pick_latest_dir(f'point{RUN_TAG}*')
RUN_DIRS['lpl'] = _pick_latest_dir(f'lpl{RUN_TAG}*')
RUN_DIRS['gau'] = _pick_latest_dir(f'gau{RUN_TAG}*')

base_run = RUN_DIRS.get(BASE_RUN_KEY) or RUN_DIRS.get('lpl') or RUN_DIRS.get('gau')
if base_run is None:
    raise RuntimeError('Could not resolve a base run for preproc_meta (point/lpl/gau missing)')
run_tag = base_run.name
train_dir = _find_training_dir_for_run_tag(run_tag)

preproc_meta = json.loads((train_dir / 'preproc_meta.json').read_text())
used_cfg = yaml.safe_load((train_dir / 'used_config.yaml').read_text())
data_cfg = (used_cfg.get('data', {}) or {})
target_col = str(data_cfg.get('target_col', 'price'))
if not FULL_CSV.exists():
    raise FileNotFoundError(f'Full dataset not found: {FULL_CSV}')

id_col = preproc_meta.get('id_col')
id_values = preproc_meta.get('id_values')
splits = (preproc_meta.get('splits', {}) or {})
if not id_col or id_values is None:
    raise RuntimeError('preproc_meta.json missing id_col/id_values; cannot build kNN OOD sets')
train_ids = np.asarray(id_values, dtype=object)[np.asarray(splits.get('train', []), dtype=int)]
test_ids = np.asarray(id_values, dtype=object)[np.asarray(splits.get('test', []), dtype=int)]

# schema
numeric_cols = list(preproc_meta.get('numeric_cols', []))
onehot_cols = list(preproc_meta.get('onehot_cols', []))
hash_cols = list(preproc_meta.get('hash_cols', []))
enc = (preproc_meta.get('encoders', {}) or {})
num_stats = (enc.get('num', {}) or {})
oh_levels = (enc.get('oh_levels', {}) or {})
hash_dims = (enc.get('hash_dims', preproc_meta.get('hash_dims', {})) or {})

# feature index offsets (as in _apply_encoders)
offset = 0
num_offset = {c: (offset + i) for i, c in enumerate(numeric_cols)}
offset += len(numeric_cols)
oh_offset, oh_index = {}, {}
for c in onehot_cols:
        levels = oh_levels.get(c, [])
        oh_offset[c] = offset
        oh_index[c] = {lv: j for j, lv in enumerate(levels)}
        offset += len(levels)
hash_offset = {}
for c in hash_cols:
    n = int(hash_dims[c])
    hash_offset[c] = offset
    offset += n
D_TOTAL = offset

P = rng.normal(0.0, 1.0 / np.sqrt(PROJ_DIM), size=(D_TOTAL, PROJ_DIM)).astype(np.float32)
hash_cache = {c: {} for c in hash_cols}

print(f'[knn-ood] run_tag={run_tag} csv={FULL_CSV} feats={D_TOTAL} -> proj={PROJ_DIM}')

ref_train_ids = rng.choice(train_ids, size=min(REF_TRAIN_N, len(train_ids)), replace=False)
ref_train_set = set(map(str, ref_train_ids))
test_set = set(map(str, test_ids))

usecols = list({id_col, target_col} | set(numeric_cols) | set(onehot_cols) | set(hash_cols) | {'year','odometer','lat','long'})

def _project_chunk(df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    Z = np.zeros((n, PROJ_DIM), dtype=np.float32)
    # numeric
    for c in numeric_cols:
        stats = num_stats.get(c)
        if stats is None:
            continue
        mean = float(stats.get('mean', 0.0)); std = float(stats.get('std', 1.0)) or 1.0
        if c in df.columns:
            x = pd.to_numeric(df[c], errors='coerce').to_numpy(float)
            x = np.where(np.isfinite(x), x, mean)
            z = ((x - mean) / std).astype(np.float32)
        else:
            z = np.zeros(n, dtype=np.float32)
        Z += z[:, None] * P[num_offset[c]][None, :]
    # onehot
    for c in onehot_cols:
        if c not in df.columns:
            continue
        idx_map = oh_index.get(c, {})
        base = oh_offset[c]
        s = df[c].astype(str).fillna('__NA__').to_numpy()
        inds = np.array([idx_map.get(v, -1) for v in s], dtype=int)
        m = inds >= 0
        if np.any(m):
            Z[m] += P[base + inds[m]]
    # hashed
    for c in hash_cols:
        if c not in df.columns:
            continue
        base = hash_offset[c]
        n_dim = int(hash_dims[c])
        cache = hash_cache[c]
        s = df[c].astype(str).fillna('__NA__').to_numpy()
        buckets = np.empty(n, dtype=int)
        signs = np.empty(n, dtype=np.float32)
        for i, v in enumerate(s):
            key = f"{c}={v}"
            if key in cache:
                b, sg = cache[key]
            else:
                b = _hash_bucket_md5(key, n_dim)
                sg = -1.0 if (_hash_bucket_md5(key + '#sign', 2) % 2) else 1.0
                cache[key] = (b, sg)
            buckets[i] = b; signs[i] = sg
        Z += signs[:, None] * P[base + buckets]
    return Z

# Pass 1: collect embeddings for ref train + test
Z_ref_list = []
test_embed = {}
seen_ref = 0
for chunk in pd.read_csv(FULL_CSV, usecols=lambda c: c in usecols, chunksize=CHUNK_ROWS):
    chunk = _add_derived(chunk, target_col)
    if chunk.empty or id_col not in chunk.columns:
        continue
    ids = chunk[id_col].astype(str)
    m_ref = ids.isin(ref_train_set)
    if m_ref.any() and seen_ref < len(ref_train_set):
        sub = chunk.loc[m_ref]
        Z = _project_chunk(sub)
        for rid, z in zip(sub[id_col].astype(str).to_numpy(), Z):
            if seen_ref < len(ref_train_set):
                Z_ref_list.append(z)
                seen_ref += 1
            else:
                break
    m_test = ids.isin(test_set)
    if m_test.any():
        sub = chunk.loc[m_test]
        Z = _project_chunk(sub)
        for tid, z in zip(sub[id_col].astype(str).to_numpy(), Z):
            test_embed[tid] = z
    if seen_ref >= len(ref_train_set) and len(test_embed) >= len(test_set):
        break

Z_ref = np.stack(Z_ref_list, axis=0)
test_ids_str = np.array(list(test_embed.keys()))
Z_test = np.stack([test_embed[k] for k in test_ids_str], axis=0)
print(f'[knn-ood] ref embeddings: {Z_ref.shape}, test embeddings: {Z_test.shape}')

# kNN scores
nn = NearestNeighbors(n_neighbors=KNN_K, algorithm='auto', metric='euclidean')
nn.fit(Z_ref)
dists, _ = nn.kneighbors(Z_test, n_neighbors=KNN_K, return_distance=True)
knn_score = dists.mean(axis=1)

mild_lo, mild_hi = np.quantile(knn_score, MILD_Q[0]), np.quantile(knn_score, MILD_Q[1])
hard_lo, hard_hi = np.quantile(knn_score, HARD_Q[0]), np.quantile(knn_score, HARD_Q[1])
id_lo, id_hi = (np.quantile(knn_score, ID_IN_Q[0]), np.quantile(knn_score, ID_IN_Q[1])) if WRITE_ID_IN else (None, None)

mild_ids = set(test_ids_str[(knn_score >= mild_lo) & (knn_score <= mild_hi)])
hard_ids = set(test_ids_str[(knn_score >= hard_lo) & (knn_score <= hard_hi)])
id_in_ids = set(test_ids_str[(knn_score >= id_lo) & (knn_score <= id_hi)]) if WRITE_ID_IN else set()

print(f'[knn-ood] mild range [{mild_lo:.4f}, {mild_hi:.4f}] n={len(mild_ids)}')
print(f'[knn-ood] hard range [{hard_lo:.4f}, {hard_hi:.4f}] n={len(hard_ids)}')
if WRITE_ID_IN:
    print(f'[knn-ood] id_in range [{id_lo:.4f}, {id_hi:.4f}] n={len(id_in_ids)}')

score_map = {k: float(v) for k, v in zip(test_ids_str, knn_score)}
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_mild = OUT_DIR / 'craigslist_ood_knn_mild.csv'
out_hard = OUT_DIR / 'craigslist_ood_knn_hard.csv'
out_idin = OUT_DIR / 'craigslist_id_knn_in.csv'

def _write_subset(out_path: Path, keep_ids: set[str]):
    wrote = False
    with out_path.open('w', newline='') as f:
        writer = None
        for chunk in pd.read_csv(FULL_CSV, chunksize=CHUNK_ROWS):
            chunk = _add_derived(chunk, target_col)
            if chunk.empty or id_col not in chunk.columns:
                continue
            ids = chunk[id_col].astype(str)
            m = ids.isin(keep_ids)
            if not m.any():
                continue
            sub = chunk.loc[m].copy()
            sub['knn_score'] = sub[id_col].astype(str).map(score_map).astype(float)
            if not wrote:
                writer = csv.writer(f)
                writer.writerow(list(sub.columns))
                wrote = True
            for row in sub.itertuples(index=False):
                writer.writerow(list(row))
    print(f'[knn-ood] wrote {out_path} rows={len(keep_ids)}')

_write_subset(out_mild, mild_ids)
_write_subset(out_hard, hard_ids)
if WRITE_ID_IN:
    _write_subset(out_idin, id_in_ids)

# quick metrics
def _summ_stats(path: Path):
    df = pd.read_csv(path, usecols=[target_col, 'knn_score'])
    return {
        'n': len(df),
        'knn_mean': float(df['knn_score'].mean()),
        'knn_q90': float(df['knn_score'].quantile(0.9)),
        'price_mean': float(df[target_col].mean()),
        'price_q90': float(df[target_col].quantile(0.9)),
    }

print('\nSummary stats:')
print('mild:', _summ_stats(out_mild))
print('hard:', _summ_stats(out_hard))
if WRITE_ID_IN:
    print('id_in:', _summ_stats(out_idin))


[knn-ood] run_tag=point_m_20251211-144003_292628 csv=/home/canelo/uncertainty_quantification/datasets/craigslist_cleaned_FULL.csv feats=4745 -> proj=32
[knn-ood] ref embeddings: (50000, 32), test embeddings: (31697, 32)
[knn-ood] mild range [2.4706, 2.7345] n=4755
[knn-ood] hard range [2.7345, 7.7385] n=1585
[knn-ood] id_in range [0.7350, 2.4706] n=25357
[knn-ood] wrote /home/canelo/uncertainty_quantification/datasets/knn_ood/craigslist_ood_knn_mild.csv rows=4755
[knn-ood] wrote /home/canelo/uncertainty_quantification/datasets/knn_ood/craigslist_ood_knn_hard.csv rows=1585
[knn-ood] wrote /home/canelo/uncertainty_quantification/datasets/knn_ood/craigslist_id_knn_in.csv rows=25357

Summary stats:
mild: {'n': 4755, 'knn_mean': 2.576024990001984, 'knn_q90': 2.6880850124359132, 'price_mean': 20288.356677181913, 'price_q90': 39554.00000000003}
hard: {'n': 1585, 'knn_mean': 2.921713415807471, 'knn_q90': 3.164184513092041, 'price_mean': 20032.835331230282, 'price_q90': 42997.4}
id_in: {'n': 25

In [2]:
# --- Spot check: do kNN OOD splits overlap train/val? ---
import json, yaml
from pathlib import Path
import pandas as pd

# Adjust if you want a different run_tag/preproc_meta
base_run = RUN_DIRS.get("point") or RUN_DIRS.get("lpl") or RUN_DIRS.get("gau")
train_dir = (PROJECT_ROOT / "outputs/trainings").rglob(base_run.name).__next__()

meta = json.loads((train_dir / "preproc_meta.json").read_text())
id_col = meta["id_col"]
id_vals = meta["id_values"]
splits = meta["splits"]
train_ids = set(str(id_vals[i]) for i in splits["train"])
val_ids   = set(str(id_vals[i]) for i in splits["val"])
test_ids  = set(str(id_vals[i]) for i in splits["test"])

paths = {
    "knn_mild": PROJECT_ROOT / "datasets/knn_ood/craigslist_ood_knn_mild.csv",
    "knn_hard": PROJECT_ROOT / "datasets/knn_ood/craigslist_ood_knn_hard.csv",
    "knn_id_in": PROJECT_ROOT / "datasets/knn_ood/craigslist_id_knn_in.csv",
}

def _id_set(path):
    df = pd.read_csv(path, usecols=[id_col])
    return set(df[id_col].astype(str).unique())

for name, p in paths.items():
    ids = _id_set(p)
    print(f"\n{name}: n={len(ids)}")
    print("  in train:", len(ids & train_ids))
    print("  in val  :", len(ids & val_ids))
    print("  in test :", len(ids & test_ids), f"({len(ids & test_ids)/len(ids):.3f} of split)")

# Optional: how much of test did we consume?
total_test = len(test_ids)
used = len((paths['knn_mild'] and _id_set(paths['knn_mild'])) | (paths['knn_hard'] and _id_set(paths['knn_hard'])))
print(f"\nTest coverage by knn_mild+hard: {used}/{total_test} = {used/total_test:.3f}")


NameError: name 'PROJECT_ROOT' is not defined

In [4]:
import json, yaml
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path('/home/canelo/uncertainty_quantification').resolve()
EVAL_ROOT = PROJECT_ROOT / 'outputs' / 'evals'
RUN_TAG = '_m_'

def _pick_latest_dir(pat):
    m = sorted(EVAL_ROOT.glob(pat))
    return m[-1] if m else None

RUN_DIRS = {
    'point': _pick_latest_dir(f'point{RUN_TAG}*'),
    'lpl':   _pick_latest_dir(f'lpl{RUN_TAG}*'),
    'gau':   _pick_latest_dir(f'gau{RUN_TAG}*'),
}

base_run = RUN_DIRS.get("point") or RUN_DIRS.get("lpl") or RUN_DIRS.get("gau")
if base_run is None:
    raise RuntimeError("No base run found under outputs/evals with RUN_TAG")

# locate matching training dir
train_root = PROJECT_ROOT / "outputs" / "trainings"
train_dir = next((p for p in train_root.rglob(base_run.name) if p.is_dir()), None)
if train_dir is None:
    raise FileNotFoundError(f"No training dir named {base_run.name} under {train_root}")

meta = json.loads((train_dir / "preproc_meta.json").read_text())
id_col = meta["id_col"]; ids = meta["id_values"]; splits = meta["splits"]
train_ids = set(str(ids[i]) for i in splits["train"])
val_ids   = set(str(ids[i]) for i in splits["val"])
test_ids  = set(str(ids[i]) for i in splits["test"])

paths = {
    "knn_mild": PROJECT_ROOT / "datasets/knn_ood/craigslist_ood_knn_mild.csv",
    "knn_hard": PROJECT_ROOT / "datasets/knn_ood/craigslist_ood_knn_hard.csv",
}

def _id_set(p):
    df = pd.read_csv(p, usecols=[id_col])
    return set(df[id_col].astype(str).unique())

for name, p in paths.items():
    ids_ood = _id_set(p)
    print(f"\n{name}: n={len(ids_ood)}")
    print("  overlap train:", len(ids_ood & train_ids))
    print("  overlap val  :", len(ids_ood & val_ids))
    print("  overlap test :", len(ids_ood & test_ids))



knn_mild: n=4755
  overlap train: 0
  overlap val  : 0
  overlap test : 4755

knn_hard: n=1585
  overlap train: 0
  overlap val  : 0
  overlap test : 1585
